# V5 H1: humanizer curriculum continuation

Run these cells from top to bottom, one at a time, on a Colab T4 GPU. H1 continues the existing V4.8 masked-mean model instead of restarting it. It never reads GRADTEX or the sealed RAID-derived benchmark.

The four continuation epochs progressively increase expert-edited AI from 25% to 50% of the Beemo-positive sampling mass while retaining human, raw-AI, LLM-edited, and PADBen examples. Epoch zero (the untouched V4.8 checkpoint) is allowed to win.

## 1. Check the GPU

If CUDA is false, choose **Runtime → Change runtime type → T4 GPU** before continuing.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Get the H1 code

This is safe in both a fresh and an existing Colab runtime.

In [ ]:
!git clone https://github.com/bonbon1235312/googlecolab-humanize-detector.git /content/humanized-ai-likelihood || true
%cd /content/humanized-ai-likelihood
!git pull -q
%cd /content/humanized-ai-likelihood/ml
!pip install -q -e .

## 3. Mount Drive and set the three paths

The input paths point to the control data and V4.8 model already in Drive. H1 writes to a new folder and does not overwrite V4.8.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
CONTROL_DATA_DIR = Path('/content/drive/MyDrive/v4-data/control-v1')
SOURCE_ARTIFACTS = Path('/content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_base')
H1_ARTIFACTS = Path('/content/drive/MyDrive/v5-artifacts/h1-humanizer-curriculum')

print('Control train file:', (CONTROL_DATA_DIR / 'train.jsonl').exists())
print('V4.8 source model:', (SOURCE_ARTIFACTS / 'model.pt').exists())
print('H1 output:', H1_ARTIFACTS)

## 4. Run the fail-fast preflight

This checks partition files, lineage isolation, tokenizer assets, the V4.8 checkpoint, and the masked-mean configuration before any GPU time is spent. It intentionally fails if the H1 output folder already contains a run; use the resume cell below in that case.

In [ ]:
import json
from humanized_detector.v5_preflight import inspect_h1_inputs

preflight = inspect_h1_inputs(CONTROL_DATA_DIR, SOURCE_ARTIFACTS, H1_ARTIFACTS)
print(json.dumps(preflight, indent=2, sort_keys=True))

## 5. Train H1

This is the main run. It prints progress every 100 batches and a subtype-aware result after each epoch. `saved_last_state=True` means the run can be resumed safely after that epoch. The best checkpoint is selected by the macro expert-edited/LLM-edited AUC, with low-FPR partial AUC and aggregate AUC as tie-breakers.

In [ ]:
!python -u -m humanized_detector.v5_train \
  --data-dir /content/drive/MyDrive/v4-data/control-v1 \
  --source-artifacts /content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_base \
  --artifacts-dir /content/drive/MyDrive/v5-artifacts/h1-humanizer-curriculum \
  --epochs 4 \
  --batch-size 64 \
  --lr 3e-5 \
  --weight-decay 0.01 \
  --label-smoothing 0.02 \
  --warmup-fraction 0.05 \
  --grad-clip-norm 1.0 \
  --progress-every 100

## Emergency resume — only if the training cell was interrupted

Reconnect to a T4, rerun cells 2 and 3, then run this cell. Do not rerun the normal preflight or fresh-training cell. Resume restores the last completed epoch, optimiser, scheduler, AMP scaler, and random-number state.

In [ ]:
# Remove the leading # characters only when recovering an interrupted run.
# !python -u -m humanized_detector.v5_train \
#   --data-dir /content/drive/MyDrive/v4-data/control-v1 \
#   --source-artifacts /content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_base \
#   --artifacts-dir /content/drive/MyDrive/v5-artifacts/h1-humanizer-curriculum \
#   --epochs 4 --batch-size 64 --lr 3e-5 --weight-decay 0.01 \
#   --label-smoothing 0.02 --warmup-fraction 0.05 --grad-clip-norm 1.0 \
#   --progress-every 100 --resume

## 6. Inspect the selected checkpoint

The summary states whether epoch zero or a curriculum epoch won. The history table lets us compare expert, LLM-edited, raw-AI, and aggregate development behaviour without touching a final benchmark.

In [ ]:
import json
import pandas as pd

summary = json.loads((H1_ARTIFACTS / 'summary.json').read_text())
history = [json.loads(line) for line in (H1_ARTIFACTS / 'history.jsonl').read_text().splitlines() if line]
display(pd.DataFrame([{
    'epoch': row['epoch'],
    'loss': row.get('training_loss'),
    'macro_edit_auc': row['selection']['macro_edit_auc'],
    'expert_auc': row['subtypes']['expert_edited_ai']['roc_auc'],
    'llm_auc': row['subtypes']['llm_edited_ai']['roc_auc'],
    'raw_auc': row['subtypes']['raw_ai']['roc_auc'],
    'aggregate_auc': row['aggregate']['roc_auc'],
    'improved': row.get('improved'),
} for row in history]))
print(json.dumps(summary, indent=2, sort_keys=True))

## 7. Freeze calibration for H1

Run this after training finishes. It uses only the existing calibration partition and writes 1%, 2%, and 5% human-FPR operating points plus cross-fitted Platt diagnostics.

In [ ]:
!python -u -m humanized_detector.v4_calibrate \
  --data-dir /content/drive/MyDrive/v4-data/control-v1 \
  --artifacts-dir /content/drive/MyDrive/v5-artifacts/h1-humanizer-curriculum

## Stop here and share the output

Do not run GRADTEX or open the sealed RAID-derived benchmark from this notebook. First compare H1 with V4.8 using development and calibration only; final benchmark evaluation remains a separate, one-time decision.